In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('data/WA_Fn-UseC_-HR-Employee-Attrition.csv')

# How many rows and columns?
print("Shape:", df.shape)

Shape: (1470, 35)


In [3]:
# See the first 5 rows
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [5]:
# See all column names
print(df.columns.tolist())

['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [7]:
# Confirm these columns are constant - useless for analysis
print("EmployeeCount unique values:", df['EmployeeCount'].unique())
print("Over18 unique values:", df['Over18'].unique())
print("StandardHours unique values:", df['StandardHours'].unique())

EmployeeCount unique values: [1]
Over18 unique values: ['Y']
StandardHours unique values: [80]


In [9]:
# Drop columns with no IAM or analytical value
columns_to_drop = [
    # Constant columns - zero variation
    'EmployeeCount',
    'Over18',
    'StandardHours',
    # Compensation columns - not relevant to IAM
    'DailyRate',
    'HourlyRate',
    'MonthlyRate',
    'MonthlyIncome',
    'PercentSalaryHike',
    'StockOptionLevel',
    # Survey/satisfaction columns - not relevant to IAM
    'EnvironmentSatisfaction',
    'JobSatisfaction',
    'JobInvolvement',
    'RelationshipSatisfaction',
    'WorkLifeBalance',
    # Other irrelevant columns
    'DistanceFromHome',
    'EducationField',
    'Education'
]

# Drop them
df_clean = df.drop(columns=columns_to_drop)

# Confirm how many columns remain
print("Original columns:", df.shape[1])
print("Remaining columns:", df_clean.shape[1])
print("Columns dropped:", df.shape[1] - df_clean.shape[1])
print("\nRemaining column names:")
print(df_clean.columns.tolist())

Original columns: 35
Remaining columns: 18
Columns dropped: 17

Remaining column names:
['Age', 'Attrition', 'BusinessTravel', 'Department', 'EmployeeNumber', 'Gender', 'JobLevel', 'JobRole', 'MaritalStatus', 'NumCompaniesWorked', 'OverTime', 'PerformanceRating', 'TotalWorkingYears', 'TrainingTimesLastYear', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [11]:
# Check for missing values in our clean dataset
print("=== MISSING VALUES CHECK ===")
missing = df_clean.isnull().sum()
print(missing)

print("\n=== DATA TYPES ===")
print(df_clean.dtypes)

=== MISSING VALUES CHECK ===
Age                        0
Attrition                  0
BusinessTravel             0
Department                 0
EmployeeNumber             0
Gender                     0
JobLevel                   0
JobRole                    0
MaritalStatus              0
NumCompaniesWorked         0
OverTime                   0
PerformanceRating          0
TotalWorkingYears          0
TrainingTimesLastYear      0
YearsAtCompany             0
YearsInCurrentRole         0
YearsSinceLastPromotion    0
YearsWithCurrManager       0
dtype: int64

=== DATA TYPES ===
Age                         int64
Attrition                  object
BusinessTravel             object
Department                 object
EmployeeNumber              int64
Gender                     object
JobLevel                    int64
JobRole                    object
MaritalStatus              object
NumCompaniesWorked          int64
OverTime                   object
PerformanceRating           int64
TotalWor

In [13]:
# Check unique values in key categorical columns
cat_columns = ['Attrition', 'Department', 'JobRole', 
               'JobLevel', 'BusinessTravel', 'OverTime']

for col in cat_columns:
    print(f"\n{col} ({df_clean[col].nunique()} unique values):")
    print(df_clean[col].value_counts())


Attrition (2 unique values):
Attrition
No     1233
Yes     237
Name: count, dtype: int64

Department (3 unique values):
Department
Research & Development    961
Sales                     446
Human Resources            63
Name: count, dtype: int64

JobRole (9 unique values):
JobRole
Sales Executive              326
Research Scientist           292
Laboratory Technician        259
Manufacturing Director       145
Healthcare Representative    131
Manager                      102
Sales Representative          83
Research Director             80
Human Resources               52
Name: count, dtype: int64

JobLevel (5 unique values):
JobLevel
1    543
2    534
3    218
4    106
5     69
Name: count, dtype: int64

BusinessTravel (3 unique values):
BusinessTravel
Travel_Rarely        1043
Travel_Frequently     277
Non-Travel            150
Name: count, dtype: int64

OverTime (2 unique values):
OverTime
No     1054
Yes     416
Name: count, dtype: int64


In [15]:
# Define which systems each department can access
# Based on real-world RBAC principles

system_mapping = {
    'Research & Development': [
        'Active Directory',
        'SAP',
        'SharePoint',
        'ServiceNow',
        'Lab Systems'
    ],
    'Sales': [
        'Active Directory',
        'Salesforce',
        'SharePoint',
        'ServiceNow',
        'CRM Portal'
    ],
    'Human Resources': [
        'Active Directory',
        'Workday',
        'SharePoint',
        'ServiceNow',
        'HR Portal'
    ]
}

# Print clearly
for dept, systems in system_mapping.items():
    print(f"\n{dept}:")
    for s in systems:
        print(f"   → {s}")


Research & Development:
   → Active Directory
   → SAP
   → SharePoint
   → ServiceNow
   → Lab Systems

Sales:
   → Active Directory
   → Salesforce
   → SharePoint
   → ServiceNow
   → CRM Portal

Human Resources:
   → Active Directory
   → Workday
   → SharePoint
   → ServiceNow
   → HR Portal


In [17]:
# Define access level for each job role
# Based on seniority and job function
# Access Levels: Read, Write, Admin

access_level_mapping = {
    # Human Resources
    'Human Resources':          'Read',

    # R&D Roles
    'Laboratory Technician':    'Read',
    'Research Scientist':       'Write',
    'Healthcare Representative':'Read',
    'Manufacturing Director':   'Write',
    'Research Director':        'Admin',

    # Sales Roles
    'Sales Representative':     'Read',
    'Sales Executive':          'Write',

    # Management (any dept)
    'Manager':                  'Write',
}

# Print clearly
print("Job Role → Access Level Mapping:")
print("="*45)
for role, level in access_level_mapping.items():
    # Add visual indicator
    if level == 'Admin':
        flag = '🔴'
    elif level == 'Write':
        flag = '🟡'
    else:
        flag = '🟢'
    print(f"{flag} {level:<8} → {role}")

Job Role → Access Level Mapping:
🟢 Read     → Human Resources
🟢 Read     → Laboratory Technician
🟡 Write    → Research Scientist
🟢 Read     → Healthcare Representative
🟡 Write    → Manufacturing Director
🔴 Admin    → Research Director
🟢 Read     → Sales Representative
🟡 Write    → Sales Executive
🟡 Write    → Manager


In [19]:
# JML Classification - Joiner, Mover, Leaver
# This is the core of any IAM programme

def classify_jml(row):
    # LEAVER - employee has left the company
    # Attrition = Yes means they resigned/were terminated
    if row['Attrition'] == 'Yes':
        return 'Leaver'
    
    # MOVER - employee changed role recently
    # YearsInCurrentRole <= 1 means recent role change
    elif row['YearsInCurrentRole'] <= 1:
        return 'Mover'
    
    # JOINER - new to the company
    # YearsAtCompany <= 1 means recently joined
    elif row['YearsAtCompany'] <= 1:
        return 'Joiner'
    
    # STABLE - no recent changes
    else:
        return 'Stable'

# Apply to every employee
df_clean['JML_Status'] = df_clean.apply(classify_jml, axis=1)

# Show the results
print("=== JML CLASSIFICATION RESULTS ===")
print(df_clean['JML_Status'].value_counts())
print(f"\nTotal employees: {len(df_clean)}")
print(f"\nPercentage breakdown:")
print((df_clean['JML_Status'].value_counts() / 
       len(df_clean) * 100).round(1).astype(str) + '%')

=== JML CLASSIFICATION RESULTS ===
JML_Status
Stable    1016
Leaver     237
Mover      217
Name: count, dtype: int64

Total employees: 1470

Percentage breakdown:
JML_Status
Stable    69.1%
Leaver    16.1%
Mover     14.8%
Name: count, dtype: object


In [25]:
df_clean['AccessLevel'] = df_clean['JobRole'].map(access_level_mapping)

# Verify it worked
print("=== ACCESS LEVEL COLUMN ADDED ===")
print(df_clean['AccessLevel'].value_counts())
print(f"\nAny unmapped roles (NaN)?")
print(df_clean['AccessLevel'].isnull().sum())
print("\nSample - first 10 rows:")
print(df_clean[['EmployeeNumber','JobRole','AccessLevel']].head(10))


=== ACCESS LEVEL COLUMN ADDED ===
AccessLevel
Write    865
Read     525
Admin     80
Name: count, dtype: int64

Any unmapped roles (NaN)?
0

Sample - first 10 rows:
   EmployeeNumber                    JobRole AccessLevel
0               1            Sales Executive       Write
1               2         Research Scientist       Write
2               4      Laboratory Technician        Read
3               5         Research Scientist       Write
4               7      Laboratory Technician        Read
5               8      Laboratory Technician        Read
6              10      Laboratory Technician        Read
7              11      Laboratory Technician        Read
8              12     Manufacturing Director       Write
9              13  Healthcare Representative        Read


In [27]:
# Risk Scoring Model
# Each factor adds points - higher score = higher risk

def calculate_risk_score(row):
    score = 0
    
    # --- ACCESS LEVEL RISK ---
    # Admin access carries highest risk
    if row['AccessLevel'] == 'Admin':
        score += 30
    elif row['AccessLevel'] == 'Write':
        score += 15
    else:  # Read
        score += 5
    
    # --- JML STATUS RISK ---
    # Leavers are highest risk - access should be gone
    if row['JML_Status'] == 'Leaver':
        score += 40
    elif row['JML_Status'] == 'Mover':
        score += 20
    elif row['JML_Status'] == 'Joiner':
        score += 10
    
    # --- TENURE RISK ---
    # Long tenure with no role change = stale access
    if row['YearsAtCompany'] > 10:
        score += 15
    elif row['YearsAtCompany'] > 5:
        score += 10
    
    # --- OVERTIME RISK ---
    # Overtime employees accessing systems outside hours
    if row['OverTime'] == 'Yes':
        score += 10
    
    # --- TRAVEL RISK ---
    # Frequent travellers = remote access risk
    if row['BusinessTravel'] == 'Travel_Frequently':
        score += 10
    elif row['BusinessTravel'] == 'Travel_Rarely':
        score += 5
    
    # --- PERFORMANCE RISK ---
    # Low performers with high access = risk
    if row['PerformanceRating'] <= 2 and row['AccessLevel'] == 'Admin':
        score += 15
    
    # --- MANAGER CHANGE RISK ---
    # Recent manager change = access review needed
    if row['YearsWithCurrManager'] <= 1:
        score += 10

    return score

# Apply risk scoring
df_clean['RiskScore'] = df_clean.apply(calculate_risk_score, axis=1)

# Classify risk level
def classify_risk(score):
    if score >= 70:
        return 'Critical'
    elif score >= 50:
        return 'High'
    elif score >= 30:
        return 'Medium'
    else:
        return 'Low'

df_clean['RiskLevel'] = df_clean['RiskScore'].apply(classify_risk)

# Show results
print("=== RISK SCORING RESULTS ===")
print(df_clean['RiskLevel'].value_counts())
print(f"\nPercentage breakdown:")
print((df_clean['RiskLevel'].value_counts() / 
       len(df_clean) * 100).round(1).astype(str) + '%')
print(f"\nRisk Score Statistics:")
print(df_clean['RiskScore'].describe().round(1))

=== RISK SCORING RESULTS ===
RiskLevel
Medium      616
Low         451
High        257
Critical    146
Name: count, dtype: int64

Percentage breakdown:
RiskLevel
Medium      41.9%
Low         30.7%
High        17.5%
Critical     9.9%
Name: count, dtype: object

Risk Score Statistics:
count    1470.0
mean       37.8
std        18.7
min         5.0
25%        25.0
50%        35.0
75%        50.0
max       100.0
Name: RiskScore, dtype: float64


In [29]:
# Expand dataset - one row per employee per system
# This mirrors how real IAM systems store access records

import pandas as pd

# Create expanded records
expanded_records = []

for _, row in df_clean.iterrows():
    # Get systems for this employee's department
    dept_systems = system_mapping.get(row['Department'], [])
    
    # Create one record per system
    for system in dept_systems:
        record = {
            'EmployeeID':            f"EMP{row['EmployeeNumber']:04d}",
            'Department':            row['Department'],
            'JobRole':               row['JobRole'],
            'JobLevel':              row['JobLevel'],
            'Age':                   row['Age'],
            'Gender':                row['Gender'],
            'SystemName':            system,
            'AccessLevel':           row['AccessLevel'],
            'JML_Status':            row['JML_Status'],
            'Attrition':             row['Attrition'],
            'YearsAtCompany':        row['YearsAtCompany'],
            'YearsInCurrentRole':    row['YearsInCurrentRole'],
            'YearsSinceLastPromotion':row['YearsSinceLastPromotion'],
            'YearsWithCurrManager':  row['YearsWithCurrManager'],
            'BusinessTravel':        row['BusinessTravel'],
            'OverTime':              row['OverTime'],
            'PerformanceRating':     row['PerformanceRating'],
            'TotalWorkingYears':     row['TotalWorkingYears'],
            'TrainingTimesLastYear': row['TrainingTimesLastYear'],
            'RiskScore':             row['RiskScore'],
            'RiskLevel':             row['RiskLevel'],
        }
        expanded_records.append(record)

# Create expanded dataframe
df_final = pd.DataFrame(expanded_records)

# Show results
print("=== EXPANDED DATASET RESULTS ===")
print(f"Original employees:    {len(df_clean)}")
print(f"Expanded access records: {len(df_final)}")
print(f"Average systems per employee: {len(df_final)/len(df_clean):.1f}")
print(f"\nColumns in final dataset: {len(df_final.columns)}")
print(f"\nRecords by Department:")
print(df_final.groupby('Department')['EmployeeID'].count())
print(f"\nRecords by Risk Level:")
print(df_final.groupby('RiskLevel')['EmployeeID'].count())

=== EXPANDED DATASET RESULTS ===
Original employees:    1470
Expanded access records: 7350
Average systems per employee: 5.0

Columns in final dataset: 21

Records by Department:
Department
Human Resources            315
Research & Development    4805
Sales                     2230
Name: EmployeeID, dtype: int64

Records by Risk Level:
RiskLevel
Critical     730
High        1285
Low         2255
Medium      3080
Name: EmployeeID, dtype: int64


In [31]:
# Export final datasets to CSV
# These files feed directly into SQL and Tableau

import os

# Create cleaned folder path
cleaned_path = 'cleaned/'

# --- Export 1: Main fact table (all access records) ---
df_final.to_csv(f'{cleaned_path}IAM_Main_AccessRecords.csv', index=False)
print(f"✅ Main access records exported: {len(df_final)} rows")

# --- Export 2: Employee summary (one row per employee) ---
df_employee_summary = df_clean[[
    'EmployeeNumber', 'Department', 'JobRole', 
    'JobLevel', 'Age', 'Gender', 'Attrition',
    'JML_Status', 'AccessLevel', 'RiskScore', 
    'RiskLevel', 'YearsAtCompany', 'YearsInCurrentRole',
    'BusinessTravel', 'OverTime', 'PerformanceRating'
]].copy()
df_employee_summary['EmployeeID'] = df_employee_summary['EmployeeNumber'].apply(
    lambda x: f"EMP{x:04d}")
df_employee_summary.to_csv(
    f'{cleaned_path}IAM_Employee_Summary.csv', index=False)
print(f"✅ Employee summary exported: {len(df_employee_summary)} rows")

# --- Export 3: Critical and High risk only ---
df_high_risk = df_final[df_final['RiskLevel'].isin(['Critical','High'])]
df_high_risk.to_csv(f'{cleaned_path}IAM_HighRisk_Records.csv', index=False)
print(f"✅ High/Critical risk records exported: {len(df_high_risk)} rows")

# --- Export 4: Leavers with active access ---
df_leavers = df_final[df_final['JML_Status'] == 'Leaver']
df_leavers.to_csv(f'{cleaned_path}IAM_Leavers_AccessRecords.csv', index=False)
print(f"✅ Leaver access records exported: {len(df_leavers)} rows")

# --- Export 5: Risk summary by department ---
df_dept_risk = df_final.groupby(['Department','RiskLevel']).agg(
    RecordCount=('EmployeeID','count'),
    AvgRiskScore=('RiskScore','mean'),
    UniqueEmployees=('EmployeeID','nunique')
).round(1).reset_index()
df_dept_risk.to_csv(f'{cleaned_path}IAM_DeptRisk_Summary.csv', index=False)
print(f"✅ Department risk summary exported: {len(df_dept_risk)} rows")

# --- Export 6: JML Summary ---
df_jml_summary = df_final.groupby(['JML_Status','RiskLevel']).agg(
    RecordCount=('EmployeeID','count'),
    AvgRiskScore=('RiskScore','mean')
).round(1).reset_index()
df_jml_summary.to_csv(f'{cleaned_path}IAM_JML_Summary.csv', index=False)
print(f"✅ JML summary exported: {len(df_jml_summary)} rows")

print(f"\n=== ALL EXPORTS COMPLETE ===")
print(f"Location: {os.path.abspath(cleaned_path)}")
print(f"\nFiles created:")
for f in os.listdir(cleaned_path):
    size = os.path.getsize(f'{cleaned_path}{f}')
    print(f"   📁 {f} ({size:,} bytes)")

✅ Main access records exported: 7350 rows
✅ Employee summary exported: 1470 rows
✅ High/Critical risk records exported: 2015 rows
✅ Leaver access records exported: 1185 rows
✅ Department risk summary exported: 12 rows
✅ JML summary exported: 9 rows

=== ALL EXPORTS COMPLETE ===
Location: /Users/neelima6/Desktop/IAM_Access_Review_Risk_Intelligence/cleaned

Files created:
   📁 IAM_HighRisk_Records.csv (251,723 bytes)
   📁 IAM_JML_Summary.csv (241 bytes)
   📁 IAM_Main_AccessRecords.csv (912,037 bytes)
   📁 IAM_DeptRisk_Summary.csv (466 bytes)
   📁 IAM_Leavers_AccessRecords.csv (149,231 bytes)
   📁 IAM_Employee_Summary.csv (159,686 bytes)


In [33]:
# PHASE 4 - STEP 1
# Create SQLite database and load our clean data

import sqlite3
import pandas as pd

# Create database in your sql folder
db_path = 'sql/IAM_AccessReview.db'
conn = sqlite3.connect(db_path)

# Load main access records into database
df_final.to_sql('AccessRecords', conn, 
                if_exists='replace', 
                index=False)

# Load employee summary into database
df_employee_summary.to_sql('EmployeeSummary', conn,
                           if_exists='replace',
                           index=False)

# Verify tables were created
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()

print("=== DATABASE CREATED SUCCESSFULLY ===")
print(f"Location: sql/IAM_AccessReview.db")
print(f"\nTables created:")
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table[0]}")
    count = cursor.fetchone()[0]
    print(f"   📋 {table[0]}: {count:,} rows")

=== DATABASE CREATED SUCCESSFULLY ===
Location: sql/IAM_AccessReview.db

Tables created:
   📋 AccessRecords: 7,350 rows
   📋 EmployeeSummary: 1,470 rows


In [35]:
# PHASE 4 - STEP 2
# SQL Query 1: Find all Leaver access records
# Business Question: Which employees left but still have system access?

query1 = """
SELECT 
    EmployeeID,
    Department,
    JobRole,
    SystemName,
    AccessLevel,
    RiskScore,
    RiskLevel,
    YearsAtCompany
FROM AccessRecords
WHERE JML_Status = 'Leaver'
ORDER BY RiskScore DESC, AccessLevel DESC
LIMIT 20;
"""

df_q1 = pd.read_sql_query(query1, conn)

print("=== QUERY 1: LEAVER ACCESS RECORDS (Top 20) ===")
print(f"Business Question: Who left the company but still has system access?")
print(f"Finding: {len(pd.read_sql_query(query1.replace('LIMIT 20',''), conn))} total leaver access records")
print(f"Action: Immediate de-provisioning required")
print()
print(df_q1.to_string(index=False))

=== QUERY 1: LEAVER ACCESS RECORDS (Top 20) ===
Business Question: Who left the company but still has system access?
Finding: 1185 total leaver access records
Action: Immediate de-provisioning required

EmployeeID             Department           JobRole       SystemName AccessLevel  RiskScore RiskLevel  YearsAtCompany
   EMP0825 Research & Development Research Director Active Directory       Admin        100  Critical              31
   EMP0825 Research & Development Research Director              SAP       Admin        100  Critical              31
   EMP0825 Research & Development Research Director       SharePoint       Admin        100  Critical              31
   EMP0825 Research & Development Research Director       ServiceNow       Admin        100  Critical              31
   EMP0825 Research & Development Research Director      Lab Systems       Admin        100  Critical              31
   EMP0492                  Sales   Sales Executive Active Directory       Write         

In [37]:
# PHASE 4 - STEP 3
# SQL Queries 2-6: Comprehensive IAM Analysis

# --- Query 2: Critical and High Risk by Department ---
query2 = """
SELECT 
    Department,
    RiskLevel,
    COUNT(*) as RecordCount,
    ROUND(AVG(RiskScore), 1) as AvgRiskScore,
    COUNT(DISTINCT EmployeeID) as UniqueEmployees
FROM AccessRecords
WHERE RiskLevel IN ('Critical', 'High')
GROUP BY Department, RiskLevel
ORDER BY Department, RiskLevel;
"""

df_q2 = pd.read_sql_query(query2, conn)
print("=== QUERY 2: HIGH/CRITICAL RISK BY DEPARTMENT ===")
print(f"Business Question: Which departments have the highest risk concentration?")
print()
print(df_q2.to_string(index=False))

print("\n" + "="*60 + "\n")

# --- Query 3: Admin Access Analysis ---
query3 = """
SELECT 
    Department,
    JobRole,
    COUNT(DISTINCT EmployeeID) as EmployeeCount,
    COUNT(*) as SystemCount,
    GROUP_CONCAT(DISTINCT SystemName) as Systems
FROM AccessRecords
WHERE AccessLevel = 'Admin'
GROUP BY Department, JobRole
ORDER BY EmployeeCount DESC;
"""

df_q3 = pd.read_sql_query(query3, conn)
print("=== QUERY 3: ADMIN ACCESS ANALYSIS ===")
print(f"Business Question: Who has Admin access and to which systems?")
print()
print(df_q3.to_string(index=False))

print("\n" + "="*60 + "\n")

# --- Query 4: Mover Access Review ---
query4 = """
SELECT 
    EmployeeID,
    Department,
    JobRole,
    AccessLevel,
    RiskScore,
    RiskLevel,
    YearsInCurrentRole,
    YearsSinceLastPromotion
FROM AccessRecords
WHERE JML_Status = 'Mover'
AND RiskLevel IN ('Critical', 'High')
GROUP BY EmployeeID, Department, JobRole, 
         AccessLevel, RiskScore, RiskLevel,
         YearsInCurrentRole, YearsSinceLastPromotion
ORDER BY RiskScore DESC
LIMIT 15;
"""

df_q4 = pd.read_sql_query(query4, conn)
print("=== QUERY 4: HIGH RISK MOVERS ===")
print(f"Business Question: Which recently moved employees need access recertification?")
print()
print(df_q4.to_string(index=False))

print("\n" + "="*60 + "\n")

# --- Query 5: Risk Score Distribution ---
query5 = """
SELECT 
    RiskLevel,
    COUNT(*) as RecordCount,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM AccessRecords), 1) as Percentage,
    ROUND(AVG(RiskScore), 1) as AvgScore,
    MIN(RiskScore) as MinScore,
    MAX(RiskScore) as MaxScore
FROM AccessRecords
GROUP BY RiskLevel
ORDER BY AVG(RiskScore) DESC;
"""

df_q5 = pd.read_sql_query(query5, conn)
print("=== QUERY 5: RISK DISTRIBUTION SUMMARY ===")
print(f"Business Question: What is the overall risk profile of our access landscape?")
print()
print(df_q5.to_string(index=False))

print("\n" + "="*60 + "\n")

# --- Query 6: JML Summary with Risk ---
query6 = """
SELECT 
    JML_Status,
    COUNT(DISTINCT EmployeeID) as UniqueEmployees,
    COUNT(*) as TotalAccessRecords,
    ROUND(AVG(RiskScore), 1) as AvgRiskScore,
    SUM(CASE WHEN RiskLevel = 'Critical' THEN 1 ELSE 0 END) as CriticalRecords,
    SUM(CASE WHEN RiskLevel = 'High' THEN 1 ELSE 0 END) as HighRecords
FROM AccessRecords
GROUP BY JML_Status
ORDER BY AvgRiskScore DESC;
"""

df_q6 = pd.read_sql_query(query6, conn)
print("=== QUERY 6: JML STATUS RISK SUMMARY ===")
print(f"Business Question: How does JML status correlate with risk levels?")
print()
print(df_q6.to_string(index=False))

# Close connection
conn.close()
print("\n✅ All queries complete. Database connection closed.")

=== QUERY 2: HIGH/CRITICAL RISK BY DEPARTMENT ===
Business Question: Which departments have the highest risk concentration?

            Department RiskLevel  RecordCount  AvgRiskScore  UniqueEmployees
       Human Resources  Critical           10          77.5                2
       Human Resources      High           55          59.1               11
Research & Development  Critical          420          75.7               84
Research & Development      High          870          55.4              174
                 Sales  Critical          300          76.5               60
                 Sales      High          360          56.1               72


=== QUERY 3: ADMIN ACCESS ANALYSIS ===
Business Question: Who has Admin access and to which systems?

            Department           JobRole  EmployeeCount  SystemCount                                                Systems
Research & Development Research Director             80          400 Active Directory,SAP,SharePoint,Service

In [39]:
# PHASE 4 - STEP 4
# Export SQL query results to CSV for Tableau

# Reconnect to database
conn = sqlite3.connect('sql/IAM_AccessReview.db')

# Re-run all queries and save results
queries = {
    'Q1_Leaver_Records': """
        SELECT EmployeeID, Department, JobRole, SystemName,
               AccessLevel, RiskScore, RiskLevel, YearsAtCompany
        FROM AccessRecords
        WHERE JML_Status = 'Leaver'
        ORDER BY RiskScore DESC
    """,
    'Q2_DeptRisk': """
        SELECT Department, RiskLevel,
               COUNT(*) as RecordCount,
               ROUND(AVG(RiskScore),1) as AvgRiskScore,
               COUNT(DISTINCT EmployeeID) as UniqueEmployees
        FROM AccessRecords
        WHERE RiskLevel IN ('Critical','High')
        GROUP BY Department, RiskLevel
    """,
    'Q3_AdminAccess': """
        SELECT Department, JobRole,
               COUNT(DISTINCT EmployeeID) as EmployeeCount,
               COUNT(*) as SystemCount,
               GROUP_CONCAT(DISTINCT SystemName) as Systems
        FROM AccessRecords
        WHERE AccessLevel = 'Admin'
        GROUP BY Department, JobRole
    """,
    'Q4_HighRiskMovers': """
        SELECT EmployeeID, Department, JobRole,
               AccessLevel, RiskScore, RiskLevel,
               YearsInCurrentRole, YearsSinceLastPromotion
        FROM AccessRecords
        WHERE JML_Status = 'Mover'
        AND RiskLevel IN ('Critical','High')
        GROUP BY EmployeeID, Department, JobRole,
                 AccessLevel, RiskScore, RiskLevel,
                 YearsInCurrentRole, YearsSinceLastPromotion
        ORDER BY RiskScore DESC
    """,
    'Q5_RiskDistribution': """
        SELECT RiskLevel,
               COUNT(*) as RecordCount,
               ROUND(COUNT(*)*100.0/(SELECT COUNT(*) 
               FROM AccessRecords),1) as Percentage,
               ROUND(AVG(RiskScore),1) as AvgScore,
               MIN(RiskScore) as MinScore,
               MAX(RiskScore) as MaxScore
        FROM AccessRecords
        GROUP BY RiskLevel
        ORDER BY AVG(RiskScore) DESC
    """,
    'Q6_JML_Summary': """
        SELECT JML_Status,
               COUNT(DISTINCT EmployeeID) as UniqueEmployees,
               COUNT(*) as TotalAccessRecords,
               ROUND(AVG(RiskScore),1) as AvgRiskScore,
               SUM(CASE WHEN RiskLevel='Critical' THEN 1 ELSE 0 END) as CriticalRecords,
               SUM(CASE WHEN RiskLevel='High' THEN 1 ELSE 0 END) as HighRecords
        FROM AccessRecords
        GROUP BY JML_Status
        ORDER BY AvgRiskScore DESC
    """
}

# Save each query result
for name, query in queries.items():
    df_result = pd.read_sql_query(query, conn)
    df_result.to_csv(f'cleaned/{name}.csv', index=False)
    print(f"✅ {name}.csv saved — {len(df_result)} rows")

conn.close()
print("\n✅ All SQL results exported to cleaned/ folder")
print("✅ Ready for Tableau!")

✅ Q1_Leaver_Records.csv saved — 1185 rows
✅ Q2_DeptRisk.csv saved — 6 rows
✅ Q3_AdminAccess.csv saved — 1 rows
✅ Q4_HighRiskMovers.csv saved — 113 rows
✅ Q5_RiskDistribution.csv saved — 4 rows
✅ Q6_JML_Summary.csv saved — 3 rows

✅ All SQL results exported to cleaned/ folder
✅ Ready for Tableau!
